**Part 3 · LangGraph** · step 17 of 23  
[← 16 First graph](16_first_graph.ipynb) · [🏠 Roadmap](../README.md) · [18 Tool-calling agent graph →](18_tool_calling_agent_graph.ipynb)

# 17 · State & Reducers — what does `Annotated[list, add_messages]` mean?

When a node returns `{"key": value}`, LangGraph must decide how to combine it with the old value.
That rule is the **reducer**.

- **No reducer** → the new value **replaces** the old one.
- **With a reducer** → `new_state = reducer(old_value, returned_value)`.

You attach a reducer with `typing.Annotated[type, reducer_function]`.
`Annotated` just attaches extra metadata to a type hint — LangGraph reads that metadata.

| Annotation | Behaviour |
|---|---|
| `count: int` | overwrite |
| `items: Annotated[list, operator.add]` | concatenate lists (`old + new`) |
| `messages: Annotated[list[AnyMessage], add_messages]` | append messages, **update in place if the id already exists**, accepts dicts/tuples |

*Runs offline — no API key needed.*

## 1. Overwrite vs. `operator.add`

In [ ]:
import operator
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END


class State(TypedDict):
    """``last`` is overwritten, ``history`` accumulates."""

    last: str
    history: Annotated[list[str], operator.add]


def step_a(state: State) -> dict:
    """Return one item for each key."""
    return {"last": "A", "history": ["A"]}


def step_b(state: State) -> dict:
    """Same keys again — watch how each one is merged."""
    return {"last": "B", "history": ["B"]}


builder = StateGraph(State)
builder.add_node("step_a", step_a)
builder.add_node("step_b", step_b)
builder.add_edge(START, "step_a")
builder.add_edge("step_a", "step_b")
builder.add_edge("step_b", END)

builder.compile().invoke({"last": "", "history": ["start"]})
# -> last == "B"  (replaced),  history == ["start", "A", "B"]  (concatenated)

## 2. `add_messages` — the reducer for chat history

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage
from langgraph.graph.message import add_messages

old = [HumanMessage("hi", id="1")]

# Appends new messages
print(add_messages(old, [AIMessage("hello!", id="2")]))

# Same id -> REPLACES that message instead of appending
print(add_messages(old, [HumanMessage("hi (edited)", id="1")]))

# Dicts are converted to message objects automatically
print(add_messages(old, [{"role": "ai", "content": "from a dict"}]))

## ⚠️ Common mistake
With `add_messages`, a node should return **only the new messages**:
```python
return {"messages": [llm.invoke(state["messages"])]}                       # ✅
return {"messages": state["messages"] + [llm.invoke(state["messages"])]}   # ❌ resends everything
```
The second version only "works" because existing ids are de-duplicated — with `operator.add` it would duplicate the whole history.

### `MessagesState` shortcut
`from langgraph.graph import MessagesState` is a ready-made state with
`messages: Annotated[list[AnyMessage], add_messages]`. Subclass it to add your own keys.

---
✅ **Done with 17 State & reducers.** Next: [18 Tool-calling agent graph →](18_tool_calling_agent_graph.ipynb)  
[🏠 Roadmap](../README.md)